# 00 Methodology And Objective Weeks

This notebook freezes the active hourly DAM methodology before any heavy benchmark execution.

What is fixed here:
- daily rolling-origin evaluation with origin at `08:00` on `D-1`
- forecast horizon `D` through `D+4`
- UTC as the internal storage timezone
- the active feature-stage ladder from `FS0` through `FS4`
- the rule that objective visual case weeks must come from **test actual prices only**

This notebook is preparation-only. It does not launch benchmark runs.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)
PACKAGE_ROOT = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from hourly_da.core.config import HourlyDAPipelineConfig
from hourly_da.core.external_features import build_external_family_catalog, load_external_feature_store
from hourly_da.core.methodology import (
    STARTER_ENDOGENOUS_FEATURE_NOTE,
    feature_stage_policy_frame,
    model_status_frame,
    shortlisting_policy_frame,
)
from hourly_da.core.reporting import find_latest_run, load_csv, load_json
from hourly_da.core.tuning import build_tuning_placeholder, tuning_cadence_frame, tuning_snippet_frame
from hourly_da.notebook_support import estimate_run_duration_seconds, format_duration, load_selected_case_weeks

config = HourlyDAPipelineConfig(
    input_csv=REPO_ROOT / "data/01_cleaned/Day_ahead_prices/DA_prices/hourly/da_prices_all_regions_hourly.csv",
    raw_root=REPO_ROOT / "data/00_Raw/DA_Prices",
    cleaned_feature_root=REPO_ROOT / "data/01_cleaned",
    output_root=REPO_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da",
)
output_root = config.output_root


def latest_run_or_none(run_label: str) -> Path | None:
    try:
        return find_latest_run(output_root, run_label)
    except FileNotFoundError:
        return None


## Active methodology snapshot

The tables below are the single source of truth for the current DAM ladder, model status, shortlisting policy, and tuning cadence.


In [ ]:
display(Markdown(f"**FS1 foundation note.** {STARTER_ENDOGENOUS_FEATURE_NOTE}"))
display(feature_stage_policy_frame())
display(model_status_frame())
display(shortlisting_policy_frame())
display(tuning_cadence_frame())


## Objective test-week selection

The selected winter, summer, and high-volatility weeks are reused later for visual diagnostics. The selection remains objective and is never hand-picked from model outputs.


In [ ]:
try:
    selected_weeks_run, selected_weeks = load_selected_case_weeks(output_root)
    print(selected_weeks_run)
    display(selected_weeks)
except FileNotFoundError:
    print("No saved objective week selection artifact exists yet.")


## Optional refresh hook


In [ ]:
ALLOW_HEAVY_RERUN = False

if ALLOW_HEAVY_RERUN:
    estimate = estimate_run_duration_seconds(output_root, "case_week_selection")
    if estimate is not None:
        print(
            "Heavy rerun warning: latest comparable run "
            f"{estimate['run_id']} suggests about {format_duration(float(estimate['estimate_seconds']))}."
        )
    else:
        print("Heavy rerun warning: no comparable runtime estimate was found for this stage.")

    command = [sys.executable, str(PACKAGE_ROOT / "run_case_week_selection.py")]

    started = time.perf_counter()
    subprocess.run(command, check=True)
    elapsed_seconds = time.perf_counter() - started
    print(f"Actual wall-clock time: {format_duration(elapsed_seconds)}")
else:
    print("Rerun skipped. Set ALLOW_HEAVY_RERUN = True only when you are ready to execute the finalized pipeline.")
